# **Artificial Neural Networks and Deep Learning**

---

## **Challenge 1**


## 🌐 **Google Drive Connection**

In [1]:
from google.colab import drive
drive.mount("/gdrive")
current_dir = "/gdrive/My\\ Drive/[2025-2026]\\ AN2DL/Challenge\\ 1"
%cd $current_dir

Mounted at /gdrive
/gdrive/My Drive/[2025-2026] AN2DL/Challenge 1


## ⚙️ **Libraries Import**

In [2]:
# Set seed for reproducibility
SEED = 42

# Import necessary libraries
import os
from sklearn.model_selection import StratifiedGroupKFold

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
import torch.nn.functional as F
# from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import TensorDataset, DataLoader
logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import copy
import shutil
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

PyTorch version: 2.8.0+cu126
Device: cuda


In [3]:
# Load dataset files
X_train = pd.read_csv('pirate_pain_train.csv')
y_train = pd.read_csv('pirate_pain_train_labels.csv')
X_test = pd.read_csv('pirate_pain_test.csv')

# Merge labels with data samples
df = X_train.merge(y_train, on="sample_index")

In [4]:
# Strip spaces
for col in ["n_legs","n_hands","n_eyes"]:
    df[col] = df[col].str.strip()

# Mapping
legs_map  = {"two": 2, "one+peg_leg": 1}
hands_map = {"two": 2, "one+hook_hand": 1}
eyes_map  = {"two": 2, "one+eye_patch": 1}

df["n_legs"]  = df["n_legs"].map(legs_map)
df["n_hands"] = df["n_hands"].map(hands_map)
df["n_eyes"]  = df["n_eyes"].map(eyes_map)

## 🔄 **Data Preprocessing**

In [5]:
from sklearn.model_selection import train_test_split

# Stratified subject-level split
subject_labels = df.groupby("sample_index")["label"].first()
subjects = subject_labels.index.values
labels = subject_labels.values

# 1) TEST = 0% stratified (all subjects for train/val)
train_val_subjects = subjects
test_subjects = np.array([]) # Create an empty array for test subjects

# Now create df_test (will be empty)
df_test = df[df["sample_index"].isin(test_subjects)].copy()

# VALIDATION = 50 samples
train_val_subjects = list(train_val_subjects)
random.seed(SEED)
random.shuffle(train_val_subjects)

N_VAL = 50
val_subjects = train_val_subjects[:N_VAL]
train_subjects = train_val_subjects[N_VAL:]

df_train = df[df["sample_index"].isin(train_subjects)].copy()
df_val   = df[df["sample_index"].isin(val_subjects)].copy()

# Create the df_no_test used for training and validation
df_no_test = df[~df["sample_index"].isin(test_subjects)].copy()

print(f"Training set shape: {df_train.shape}")
print(f"Validation set shape: {df_val.shape}")
print(f"Test set shape: {df_test.shape}")

Training set shape: (97760, 41)
Validation set shape: (8000, 41)
Test set shape: (0, 41)


In [6]:
# Initialize the class dictionary
training_labels = {
    'no_pain': 0,
    'low_pain': 0,
    'high_pain': 0
}

# Count occurrences of each label for a unique sequence (sample_index)
for sample_idx in df_train['sample_index'].unique():
    label = df_train[df_train['sample_index'] == sample_idx]['label'].values[0]
    training_labels[label] += 1

# Print the class distribution in the training set
print('Class distribution in the training and validation set:', training_labels)

Class distribution in the training and validation set: {'no_pain': 469, 'low_pain': 89, 'high_pain': 53}


In [7]:
# Initialize the class dictionary
val_labels = {
    'no_pain': 0,
    'low_pain': 0,
    'high_pain': 0
}

# Count occurrences of each label for a unique sequence (sample_index)
for sample_idx in df_val['sample_index'].unique():
    label = df_val[df_val['sample_index'] == sample_idx]['label'].values[0]
    val_labels[label] += 1

# Print the class distribution in the validation set
print('Distribuzione delle classi nel validation set:', val_labels)


Distribuzione delle classi nel validation set: {'no_pain': 42, 'low_pain': 5, 'high_pain': 3}


In [8]:
# Print the class distribution in the training set
label_mapping = {
    'no_pain': 0,
    'low_pain': 1,
    'high_pain': 2
}

# Apply the mapping to the three sets
df_train['label'] = df_train['label'].map(label_mapping).astype(int)
df_val['label'] = df_val['label'].map(label_mapping).astype(int)
df_test['label'] = df_test['label'].map(label_mapping).astype(int)

In [9]:
scale_columns = ['pain_survey_1','pain_survey_2','pain_survey_3','pain_survey_4'] + [f'joint_{i:02d}' for i in range(31)]

# Min and max from the training set
mins = df_train[scale_columns].min()
maxs = df_train[scale_columns].max()

# Min-Max Normalization
for col in scale_columns:
    if maxs[col] == mins[col]:
        # Handle constant columns by setting them to 0 after normalization
        df_train[col] = 0.0
        df_val[col] = 0.0
        df_test[col] = 0.0
    else:
        df_train[col] = (df_train[col] - mins[col]) / (maxs[col] - mins[col])
        df_val[col] = (df_val[col] - mins[col]) / (maxs[col] - mins[col])
        df_test[col] = (df_test[col] - mins[col]) / (maxs[col] - mins[col])

In [10]:
# Define the window size
WINDOW_SIZE = 44

# Define the stride for overlapping windows
STRIDE = 22

In [11]:
# Dynamic feature columns - not include the static ones (n_legs, n_hands, n_eyes)
feature_columns = ['pain_survey_1','pain_survey_2','pain_survey_3','pain_survey_4'] + [f'joint_{i:02d}' for i in range(31)]

# Define a function to build sequences from the dataset
def build_sequences(df, window=200, stride=200):
    # Sanity check to ensure the window is divisible by the stride
    assert window % stride == 0

    # Initialise lists to store sequences and their corresponding labels
    dataset = []
    labels = []
    static_list = []

    # Iterate over unique IDs in the DataFrame
    for sample_id in df['sample_index'].unique():
        # Extract sensor data for the current sample
        temp = df[df['sample_index'] == sample_id][feature_columns].values

        # Retrieve the activity label for the current ID
        label = df[df['sample_index'] == sample_id]['label'].values[0]  # prendi la label del soggetto

        # Get static features for this sample_id
        static_feats = df[df['sample_index'] == sample_id][['n_legs','n_hands','n_eyes']].values[0]

        # Calculate padding length to ensure full windows
        padding_len = (window - len(temp) % window) % window

        # Create zero padding and concatenate with the data
        padding = np.zeros((padding_len, len(feature_columns)), dtype='float32')
        temp = np.concatenate((temp, padding))

        # Build feature windows and associate them with labels
        idx = 0
        while idx + window <= len(temp):
            dataset.append(temp[idx:idx + window])
            labels.append(label)
            static_list.append(static_feats)
            idx += stride

    # Convert lists to numpy arrays for further processing
    dataset = np.array(dataset)
    labels = np.array(labels)

    return np.array(dataset), np.array(labels), np.array(static_list)

In [12]:
X_train, y_train, static_train = build_sequences(df_train, WINDOW_SIZE, STRIDE)
X_val, y_val, static_val = build_sequences(df_val,   WINDOW_SIZE, STRIDE)
X_test, y_test, static_test = build_sequences(df_test,  WINDOW_SIZE, STRIDE)

# Print the shapes of the generated datasets and their labels
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((4277, 44, 35), (4277,), (350, 44, 35), (350,), (0,), (0,))

In [13]:
# Define the input shape based on the training data
input_shape = X_train.shape[1:]

# Define the number of classes based on the categorical labels
num_classes = len(np.unique(y_train))

In [14]:
# Convert numpy arrays to PyTorch datasets (pairs features with labels)
train_ds = TensorDataset(
    torch.from_numpy(X_train.astype(np.float32)),
    torch.from_numpy(static_train.astype(np.int64)),    # static features
    torch.from_numpy(y_train)
)
val_ds   = TensorDataset(
    torch.from_numpy(X_val.astype(np.float32)),
    torch.from_numpy(static_val.astype(np.int64)),      # static features
    torch.from_numpy(y_val)
)
test_ds  = TensorDataset(
    torch.from_numpy(X_test.astype(np.float32)),
    torch.from_numpy(static_test.astype(np.int64)),     # static features
    torch.from_numpy(y_test)
)

In [15]:
# Define the batch size, which is the number of samples in each batch
BATCH_SIZE = 64

In [16]:
def make_loader(ds, batch_size, shuffle, drop_last):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    # Create DataLoader with performance optimizations
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,  # Faster GPU transfer
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=4,  # Load 4 batches ahead
    )

In [17]:
# Create data loaders with different settings for each phase
train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = make_loader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = make_loader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

## 🛠️ **Model Building**

In [18]:
def recurrent_summary(model, input_size):
    """
    Custom summary function that emulates torchinfo's output while correctly
    counting parameters for RNN/GRU/LSTM layers.

    This function is designed for models whose direct children are
    nn.Linear, nn.RNN, nn.GRU, or nn.LSTM layers.

    Args:
        model (nn.Module): The model to analyze.
        input_size (tuple): Shape of the input tensor (e.g., (seq_len, features)).
    """

    # Dictionary to store output shapes captured by forward hooks
    output_shapes = {}
    # List to track hook handles for later removal
    hooks = []

    def get_hook(name):
        """Factory function to create a forward hook for a specific module."""
        def hook(module, input, output):
            # Handle RNN layer outputs (returns a tuple)
            if isinstance(output, tuple):
                # output[0]: all hidden states with shape (batch, seq_len, hidden*directions)
                shape1 = list(output[0].shape)
                shape1[0] = -1  # Replace batch dimension with -1

                # output[1]: final hidden state h_n (or tuple (h_n, c_n) for LSTM)
                if isinstance(output[1], tuple):  # LSTM case: (h_n, c_n)
                    shape2 = list(output[1][0].shape)  # Extract h_n only
                else:  # RNN/GRU case: h_n only
                    shape2 = list(output[1].shape)

                # Replace batch dimension (middle position) with -1
                shape2[1] = -1

                output_shapes[name] = f"[{shape1}, {shape2}]"

            # Handle standard layer outputs (e.g., Linear)
            else:
                shape = list(output.shape)
                shape[0] = -1  # Replace batch dimension with -1
                output_shapes[name] = f"{shape}"
        return hook

    # 1. Determine the device where model parameters reside
    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = torch.device("cpu")  # Fallback for models without parameters

    # 2. Create a dummy input tensor with batch_size=1
    dummy_input = torch.randn(1, *input_size).to(device)

    # 3. Register forward hooks on target layers
    # Iterate through direct children of the model (e.g., self.rnn, self.classifier)
    for name, module in model.named_children():
        if isinstance(module, (nn.Linear, nn.RNN, nn.GRU, nn.LSTM)):
            # Register the hook and store its handle for cleanup
            hook_handle = module.register_forward_hook(get_hook(name))
            hooks.append(hook_handle)

    # 4. Execute a dummy forward pass in evaluation mode
    model.eval()
    with torch.no_grad():
        try:
            model(dummy_input)
        except Exception as e:
            print(f"Error during dummy forward pass: {e}")
            # Clean up hooks even if an error occurs
            for h in hooks:
                h.remove()
            return

    # 5. Remove all registered hooks
    for h in hooks:
        h.remove()

    # --- 6. Print the summary table ---

    print("-" * 79)
    # Column headers
    print(f"{'Layer (type)':<25} {'Output Shape':<28} {'Param #':<18}")
    print("=" * 79)

    total_params = 0
    total_trainable_params = 0

    # Iterate through modules again to collect and display parameter information
    for name, module in model.named_children():
        if name in output_shapes:
            # Count total and trainable parameters for this module
            module_params = sum(p.numel() for p in module.parameters())
            trainable_params = sum(p.numel() for p in module.parameters() if p.requires_grad)

            total_params += module_params
            total_trainable_params += trainable_params

            # Format strings for display
            layer_name = f"{name} ({type(module).__name__})"
            output_shape_str = str(output_shapes[name])
            params_str = f"{trainable_params:,}"

            print(f"{layer_name:<25} {output_shape_str:<28} {params_str:<15}")

    print("=" * 79)
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {total_trainable_params:,}")
    print(f"Non-trainable params: {total_params - total_trainable_params:,}")
    print("-" * 79)

In [19]:
# Recurrent Neural Network (RNN) classifier with an attention mechanism.
# It combines dynamic time-series features with static embeddings to predict classes.

class RecurrentClassifierWithAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes,
                 rnn_type='GRU', bidirectional=False, dropout_rate=0.1):
        super().__init__()

        self.bidirectional = bidirectional
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        rnn_map = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}
        RNN = rnn_map[rnn_type]

        # Create the recurrent layer
        self.rnn = RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout_rate if num_layers > 1 else 0
        )

        rnn_output_dim = hidden_size * (2 if bidirectional else 1)

        # Attention
        self.attn = AttentionEncoderOnly(rnn_output_dim)

        # Embedding dimension
        self.static_emb_dim = 2
        self.emb_legs  = nn.Embedding(num_embeddings=3, embedding_dim=self.static_emb_dim)
        self.emb_hands = nn.Embedding(num_embeddings=3, embedding_dim=self.static_emb_dim)
        self.emb_eyes  = nn.Embedding(num_embeddings=3, embedding_dim=self.static_emb_dim)

        # Project static embeddings to a meaningful size and add dropout
        self.static_proj = nn.Sequential(
            nn.Linear(self.static_emb_dim * 3, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Final classifier
        self.classifier = nn.Linear(rnn_output_dim + hidden_size // 2, num_classes)

    def forward(self, x, static_feats):
        encoder_outputs, hidden = self.rnn(x)

        # Handle LSTM hidden state
        if isinstance(hidden, tuple):
            hidden = hidden[0]

        # Select last layer's hidden state(s)
        # For bidirectional, the last two items in 'hidden' are the last layer's forward and backward states.
        if self.bidirectional:
            # Concatenate the last layer's forward and backward hidden states
            hidden_last = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        else:
            # For unidirectional, just take the last layer's hidden state
            hidden_last = hidden[-1, :, :]

        context, attn_weights = self.attn(encoder_outputs, hidden_last)

        # ---- Static embeddings ----
        legs_emb  = self.emb_legs(static_feats[:, 0])
        hands_emb = self.emb_hands(static_feats[:, 1])
        eyes_emb  = self.emb_eyes(static_feats[:, 2])

        static_emb = torch.cat([legs_emb, hands_emb, eyes_emb], dim=1)

        # Project static features
        static_proj = self.static_proj(static_emb)

        # ---- Fuse temporal context + static information ----
        combined = torch.cat([context, static_proj], dim=1)

        logits = self.classifier(combined)
        return logits

## 🧮 **Network and Training Hyperparameters**

In [20]:
# Training configuration
LEARNING_RATE = 1e-3
EPOCHS = 1500
PATIENCE = 250

# Architecture
HIDDEN_LAYERS = 2       # Hidden layers
HIDDEN_SIZE = 128       # Neurons per layer

# Regularisation
DROPOUT_RATE = 0.5      # Dropout probability
L1_LAMBDA = 0           # L1 penalty
L2_LAMBDA = 1e-4        # L2 penalty

# Weighted CrossEntropyLoss
# Convert to tensor
class_counts = torch.tensor(list(training_labels.values()), dtype=torch.float32)

# Calculate inverse frequency weights
class_weights = 1.0 / class_counts

# Normalize to sum to the number of classes
class_weights = class_weights / class_weights.sum() * len(class_counts)

print("Class weights:", class_weights)

# Pass weights to CrossEntropyLoss
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

Class weights: tensor([0.1984, 1.0457, 1.7559])


## 🧠 **Model Training**

In [21]:
# Initialize best model tracking variables
best_model = None
best_performance = float('-inf')

In [22]:
def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
    """
    Perform one complete training epoch through the entire training dataset.

    Args:
        model (nn.Module): The neural network model to train
        train_loader (DataLoader): PyTorch DataLoader containing training data batches
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss)
        optimizer (torch.optim): Optimization algorithm (e.g., Adam, SGD)
        scaler (GradScaler): PyTorch's gradient scaler for mixed precision training
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)
        l1_lambda (float): Lambda for L1 regularization
        l2_lambda (float): Lambda for L2 regularization

    Returns:
        tuple: (average_loss, f1 score) - Training loss and f1 score for this epoch
    """
    model.train()  # Set model to training mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Iterate through training batches
    for batch_idx, (inputs, static_feats, targets) in enumerate(train_loader):
        # Move data to device (GPU/CPU)
        inputs, targets, static_feats = inputs.to(device), targets.to(device), static_feats.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with mixed precision (if CUDA available)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs, static_feats)
            loss = criterion(logits, targets)

            # Add L1 and L2 regularization
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            l2_norm = sum(p.pow(2).sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm


        # Backward pass with gradient scaling
        scaler.scale(loss).backward()

        # Unscale + clip to prevent exploding gradients
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        running_loss += loss.item() * inputs.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_f1

In [23]:
def validate_one_epoch(model, val_loader, criterion, device):
    """
    Perform one complete validation epoch through the entire validation dataset.

    Args:
        model (nn.Module): The neural network model to evaluate (must be in eval mode)
        val_loader (DataLoader): PyTorch DataLoader containing validation data batches
        criterion (nn.Module): Loss function used to calculate validation loss
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)

    Returns:
        tuple: (average_loss, accuracy) - Validation loss and accuracy for this epoch

    Note:
        This function automatically sets the model to evaluation mode and disables
        gradient computation for efficiency during validation.
    """
    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Disable gradient computation for validation
    with torch.no_grad():
        for inputs, static_feats, targets in val_loader:
            # Move data to device
            inputs, targets, static_feats = inputs.to(device), targets.to(device), static_feats.to(device)

            # Forward pass with mixed precision (if CUDA available)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs, static_feats)
                loss = criterion(logits, targets)

            # Accumulate metrics
            running_loss += loss.item() * inputs.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_accuracy = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_accuracy

In [24]:
def log_metrics_to_tensorboard(writer, epoch, train_loss, train_f1, val_loss, val_f1, model):
    """
    Log training metrics and model parameters to TensorBoard for visualization.

    Args:
        writer (SummaryWriter): TensorBoard SummaryWriter object for logging
        epoch (int): Current epoch number (used as x-axis in TensorBoard plots)
        train_loss (float): Training loss for this epoch
        train_f1 (float): Training f1 score for this epoch
        val_loss (float): Validation loss for this epoch
        val_f1 (float): Validation f1 score for this epoch
        model (nn.Module): The neural network model (for logging weights/gradients)

    Note:
        This function logs scalar metrics (loss/f1 score) and histograms of model
        parameters and gradients, which helps monitor training progress and detect
        issues like vanishing/exploding gradients.
    """
    # Log scalar metrics
    writer.add_scalar('Loss/Training', train_loss, epoch)
    writer.add_scalar('Loss/Validation', val_loss, epoch)
    writer.add_scalar('F1/Training', train_f1, epoch)
    writer.add_scalar('F1/Validation', val_f1, epoch)

    # Log model parameters and gradients
    for name, param in model.named_parameters():
        if param.requires_grad:
            # Check if the tensor is not empty before adding a histogram
            if param.numel() > 0:
                writer.add_histogram(f'{name}/weights', param.data, epoch)
            if param.grad is not None:
                # Check if the gradient tensor is not empty before adding a histogram
                if param.grad.numel() > 0:
                    if param.grad is not None and torch.isfinite(param.grad).all():
                        writer.add_histogram(f'{name}/gradients', param.grad.data, epoch)

In [25]:
def fit(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
        l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode='max',
        restore_best_weights=True, writer=None, verbose=10, experiment_name=""):
    """
    Train the neural network model on the training data and validate on the validation data.

    Args:
        model (nn.Module): The neural network model to train
        train_loader (DataLoader): PyTorch DataLoader containing training data batches
        val_loader (DataLoader): PyTorch DataLoader containing validation data batches
        epochs (int): Number of training epochs
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss)
        optimizer (torch.optim): Optimization algorithm (e.g., Adam, SGD)
        scaler (GradScaler): PyTorch's gradient scaler for mixed precision training
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)
        l1_lambda (float): L1 regularization coefficient (default: 0)
        l2_lambda (float): L2 regularization coefficient (default: 0)
        patience (int): Number of epochs to wait for improvement before early stopping (default: 0)
        evaluation_metric (str): Metric to monitor for early stopping (default: "val_f1")
        mode (str): 'max' for maximizing the metric, 'min' for minimizing (default: 'max')
        restore_best_weights (bool): Whether to restore model weights from best epoch (default: True)
        writer (SummaryWriter, optional): TensorBoard SummaryWriter object for logging (default: None)
        verbose (int, optional): Frequency of printing training progress (default: 10)
        experiment_name (str, optional): Experiment name for saving models (default: "")

    Returns:
        tuple: (model, training_history) - Trained model and metrics history
    """

    # Initialize metrics tracking
    training_history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': []
    }

    # Configure early stopping if patience is set
    if patience > 0:
        patience_counter = 0
        best_metric = float('-inf') if mode == 'max' else float('inf')
        best_epoch = 0

    # Ensure criterion is on the correct device
    criterion = criterion.to(device)

    print(f"Training {epochs} epochs...")

    # Main training loop: iterate through epochs
    for epoch in range(1, epochs + 1):

        # Forward pass through training data, compute gradients, update weights
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )

        # Evaluate model on validation data without updating weights
        val_loss, val_f1 = validate_one_epoch(
            model, val_loader, criterion, device
        )

        # Store metrics for plotting and analysis
        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)
        training_history['train_f1'].append(train_f1)
        training_history['val_f1'].append(val_f1)

        # Write metrics to TensorBoard for visualization
        if writer is not None:
            log_metrics_to_tensorboard(
                writer, epoch, train_loss, train_f1, val_loss, val_f1, model
            )

        # Print progress every N epochs or on first epoch
        if verbose > 0:
            if epoch % verbose == 0 or epoch == 1:
                print(f"Epoch {epoch:3d}/{epochs} | "
                    f"Train: Loss={train_loss:.4f}, F1 Score={train_f1:.4f} | "
                    f"Val: Loss={val_loss:.4f}, F1 Score={val_f1:.4f}")

        # Early stopping logic: monitor metric and save best model
        if patience > 0:
            current_metric = training_history[evaluation_metric][-1]
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)

            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered after {epoch} epochs.")
                    break

    # Restore best model weights if early stopping was used
    if restore_best_weights and patience > 0:
        model.load_state_dict(torch.load("models/"+experiment_name+'_model.pt'))
        print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")

    # Save final model if no early stopping
    if patience == 0:
        torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')

    # Close TensorBoard writer
    if writer is not None:
        writer.close()

    return model, training_history, best_metric

In [26]:
# Attention mechanism for encoder-only sequence classification.
# It calculates attention scores to create a context vector, allowing the model to focus on important parts of the input sequence.

class AttentionEncoderOnly(nn.Module):
    """
    Attention layer adapted for encoder-only sequence classification.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim * 2, 1)

    def forward(self, encoder_outputs, last_hidden):
        """
        encoder_outputs: (batch, seq_len, hidden_dim)
        last_hidden:     (batch, hidden_dim)
        """
        batch_size, seq_len, hidden_dim = encoder_outputs.size()

        # Repeat last hidden state across the sequence
        last_hidden_expanded = last_hidden.unsqueeze(1).repeat(1, seq_len, 1)

        # Concatenate
        combined = torch.cat([last_hidden_expanded, encoder_outputs], dim=2)

        # Score per timestep
        scores = self.attention(combined).squeeze(-1)  # (batch, seq_len)

        # Softmax over timesteps
        attn_weights = F.softmax(scores, dim=1)

        # Weighted sum of encoder outputs
        context = torch.sum(attn_weights.unsqueeze(-1) * encoder_outputs, dim=1)

        return context, attn_weights

## **K-Shuffle-Split Cross Validation**


In [27]:
import torch
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
import copy

# K-fold stratified group cross-validation.
# It splits data by subjects, trains models, and evaluates their performance across folds.
def k_stratified_group_cross_validation_rnn(df, epochs, criterion, device,
                            k, batch_size, hidden_layers, hidden_size, learning_rate, dropout_rate,
                            window_size, stride, rnn_type, bidirectional,
                            l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode='max',
                            restore_best_weights=True, writer=None, verbose=10, seed=42, experiment_name=""):
    """
    Perform K-fold Stratified Group cross-validation with user-based splitting.
    """

    # Initialise containers for results
    fold_losses = {}
    fold_metrics = {}
    best_scores = {}

    feature_cols = ['pain_survey_1','pain_survey_2','pain_survey_3','pain_survey_4'] + [f'joint_{i:02d}' for i in range(31)]
    in_features = len(feature_cols)

    # Assume num_classes is 3 based on 'no_pain', 'low_pain', 'high_pain'
    num_classes = 3

    # Initialise model architecture
    model = RecurrentClassifierWithAttention(
        input_size=in_features,
        hidden_size=hidden_size,
        num_layers=hidden_layers,
        num_classes=num_classes,
        dropout_rate=dropout_rate,
        bidirectional=bidirectional,
        rnn_type=rnn_type
    ).to(device)

    # Store initial weights
    initial_state = copy.deepcopy(model.state_dict())

    # Initialize ensemble and model A trackers
    best_score_A = -1.0
    best_model_A = None
    all_best_models = [] # List of best models for each split (Ensemble)

    # --- StratifiedGroupKFold Setup ---
    subject_labels = df.groupby('sample_index')['label'].first()
    groups = subject_labels.index.values
    y_subjects = subject_labels.values
    X_placeholder = np.zeros(len(groups))
    sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)

    # Iterate through K stratified, grouped splits
    for split_idx, (train_indices, val_indices) in enumerate(sgkf.split(X_placeholder, y_subjects, groups)):

        if verbose > 0:
            print(f"Split {split_idx+1}/{k}")

        train_subjects = groups[train_indices]
        val_subjects = groups[val_indices]

        df_train = df[df['sample_index'].isin(train_subjects)].copy()
        df_val = df[df['sample_index'].isin(val_subjects)].copy()

        for col in ["n_legs", "n_hands", "n_eyes"]:
            # If they are strings, convert to mapped ints
            if df_train[col].dtype == object:
                df_train[col] = df_train[col].str.strip().map(legs_map if col=="n_legs" else hands_map if col=="n_hands" else eyes_map)

            if df_val[col].dtype == object:
                df_val[col] = df_val[col].str.strip().map(legs_map if col=="n_legs" else hands_map if col=="n_hands" else eyes_map)

            df_train[col] = df_train[col].astype(int)
            df_val[col]   = df_val[col].astype(int)

        extra_cols_to_drop = ['sample_time', 'sample_label', 'id']
        for col in extra_cols_to_drop:
            if col in df_train.columns:
                df_train = df_train.drop(columns=[col])
            if col in df_val.columns:
                df_val = df_val.drop(columns=[col])

        label_mapping = {
            'no_pain': 0,
            'low_pain': 1,
            'high_pain': 2
        }

        df_train['label'] = df_train['label'].map(label_mapping).astype(int)
        df_val['label'] = df_val['label'].map(label_mapping).astype(int)

        scale_columns = ['pain_survey_1','pain_survey_2','pain_survey_3','pain_survey_4'] + [f'joint_{i:02d}' for i in range(31)]
        mins = df_train[scale_columns].min()
        maxs = df_train[scale_columns].max()

        for col in scale_columns:
            if maxs[col] == mins[col]:
                df_train[col] = 0.0
                df_val[col] = 0.0
            else:
                df_train[col] = (df_train[col] - mins[col]) / (maxs[col] - mins[col])
                df_val[col] = (df_val[col] - mins[col]) / (maxs[col] - mins[col])

        if verbose > 0:
            print(f"  Training set shape: {df_train.shape} ({len(train_subjects)} subjects)")
            print(f"  Validation set shape: {df_val.shape} ({len(val_subjects)} subjects)")

        X_train, y_train, static_train = build_sequences(df_train, WINDOW_SIZE, STRIDE)
        X_val, y_val, static_val = build_sequences(df_val,   WINDOW_SIZE, STRIDE)

        if verbose > 0:
            print(f"  Training sequences shape: {X_train.shape}")
            print(f"  Validation sequences shape: {X_val.shape}")

        train_ds = TensorDataset(
        torch.from_numpy(X_train.astype(np.float32)),
        torch.from_numpy(static_train.astype(np.int64)),    # static features
        torch.from_numpy(y_train)
        )
        val_ds   = TensorDataset(
            torch.from_numpy(X_val.astype(np.float32)),
            torch.from_numpy(static_val.astype(np.int64)),      # static features
            torch.from_numpy(y_val)
        )


        train_loader = make_loader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
        val_loader   = make_loader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)

        model.load_state_dict(initial_state)

        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=l2_lambda)
        split_scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
        os.makedirs(f"models/{experiment_name}", exist_ok=True)

        model, training_history, score = fit(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=epochs,
            criterion=criterion,
            optimizer=optimizer,
            scaler=split_scaler,
            device=device,
            writer=writer,
            patience=patience,
            verbose=verbose,
            l1_lambda=l1_lambda,
            evaluation_metric=evaluation_metric,
            mode=mode,
            restore_best_weights=restore_best_weights,
            experiment_name=experiment_name+"/split_"+str(split_idx)
        )

        # Store results
        fold_losses[f"split_{split_idx}"] = training_history['val_loss']
        fold_metrics[f"split_{split_idx}"] = training_history['val_f1']

        best_val_f1_split = max(training_history['val_f1'])
        best_scores[f"split_{split_idx}"] = best_val_f1_split

        # Track ensemble and model A
        # Capture the model from this split (it has the best weights for this fold)
        all_best_models.append(copy.deepcopy(model))

        # Check if this model is the overall best model (Model A)
        if best_val_f1_split > best_score_A:
            best_score_A = best_val_f1_split
            best_model_A = copy.deepcopy(model)

    # Compute mean and standard deviation
    best_scores["mean"] = np.mean([best_scores[k] for k in best_scores.keys() if k.startswith("split_")])
    best_scores["std"] = np.std([best_scores[k] for k in best_scores.keys() if k.startswith("split_")])

    if verbose > 0:
        print(f"Best score: {best_scores['mean']:.4f}±{best_scores['std']:.4f}")

    # The function will return 5 items
    return fold_losses, fold_metrics, best_scores, best_model_A, all_best_models

In [28]:
%%time
# Create model and display architecture with parameter count
fold_losses, fold_metrics, best_scores, best_model_A, all_best_models = k_stratified_group_cross_validation_rnn(
    df=df_no_test,
    epochs=EPOCHS,
    criterion=criterion,
    device=device,
    k=5,
    batch_size=BATCH_SIZE,
    hidden_layers=HIDDEN_LAYERS,
    hidden_size=HIDDEN_SIZE,
    learning_rate=LEARNING_RATE,
    dropout_rate=DROPOUT_RATE,
    l1_lambda=L1_LAMBDA,
    l2_lambda=L2_LAMBDA,
    verbose=1,
    patience=PATIENCE,
    seed=SEED,
    experiment_name="gru_baseline",
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    rnn_type='GRU',
    bidirectional=False
)



Split 1/5
  Training set shape: (84480, 41) (528 subjects)
  Validation set shape: (21280, 41) (133 subjects)
  Training sequences shape: (3696, 44, 35)
  Validation sequences shape: (931, 44, 35)
Training 1500 epochs...
Epoch   1/1500 | Train: Loss=1.2081, F1 Score=0.3568 | Val: Loss=1.2651, F1 Score=0.4724
Epoch   2/1500 | Train: Loss=1.1300, F1 Score=0.5904 | Val: Loss=1.2448, F1 Score=0.2407
Epoch   3/1500 | Train: Loss=1.1148, F1 Score=0.5354 | Val: Loss=1.3086, F1 Score=0.4414
Epoch   4/1500 | Train: Loss=1.0717, F1 Score=0.6437 | Val: Loss=1.2211, F1 Score=0.6369
Epoch   5/1500 | Train: Loss=1.0351, F1 Score=0.7255 | Val: Loss=1.1427, F1 Score=0.7728
Epoch   6/1500 | Train: Loss=0.9827, F1 Score=0.7539 | Val: Loss=1.0953, F1 Score=0.8394
Epoch   7/1500 | Train: Loss=0.9743, F1 Score=0.7285 | Val: Loss=1.1373, F1 Score=0.8020
Epoch   8/1500 | Train: Loss=0.9501, F1 Score=0.7702 | Val: Loss=0.9999, F1 Score=0.8454
Epoch   9/1500 | Train: Loss=0.9171, F1 Score=0.7773 | Val: Loss=0.

In [29]:
import torch
import torch.nn.functional as F
import numpy as np

# Soft Voting Ensemble function. It averages class probabilities from multiple models.
# For each input sequence, then selects the class with the highest average probability as the final prediction.
def predict_with_ensemble(model_list, data_loader, device):
    """
    Performs prediction using a Soft Voting Ensemble of K models.

    Args:
        model_list (list): A list containing the K best-performing models
                           from each cross-validation split.
        data_loader (DataLoader): DataLoader for the test or submission data.
        device (torch.device): The device to run the predictions on ('cuda' or 'cpu').

    Returns:
        np.array: The final predicted class labels (0, 1, or 2).
    """
    if not model_list:
        raise ValueError("Model list cannot be empty.")

    # Initialize total probability accumulator
    all_probabilities = []

    # Iterate through all K models
    for model_idx, model in enumerate(model_list):
        model.eval()  # Set model to evaluation mode

        # Accumulator for this specific model's probabilities
        model_probabilities = []

        with torch.no_grad():
            for batch_data in data_loader:
                # The DataLoader yields (features, static_features, optional_labels)
                # Features (index 0) and static_features (index 1)
                X_batch = batch_data[0].to(device)
                static_batch = batch_data[1].to(device) # Static features are at index 1

                # Get the raw output (logits)
                outputs = model(X_batch, static_batch) # Pass both arguments

                # Convert logits to probabilities using Softmax
                probs = F.softmax(outputs, dim=1).cpu().numpy()
                model_probabilities.append(probs)

        # Concatenate all batch probabilities for this model
        all_probabilities.append(np.concatenate(model_probabilities, axis=0))

    # Ensemble (Soft Voting)
    # Check if all models produced the same number of predictions
    if not all(p.shape == all_probabilities[0].shape for p in all_probabilities):
        raise RuntimeError("Prediction output shapes do not match across models.")

    # Average the probabilities across all K models (axis=0 is the model index when stacked)
    # The result is (num_sequences, num_classes)
    avg_probabilities = np.mean(all_probabilities, axis=0);

    # Final Prediction
    # The predicted class is the one with the highest average probability
    ensemble_predictions = np.argmax(avg_probabilities, axis=1)

    return ensemble_predictions

In [30]:
# This function performs Test-Time Augmentation (TTA) with soft voting ensemble.
# It processes batches with augmented versions (shifts, noise) across multiple models, averaging probabilities for robust predictions.
def predict_with_tta_ensemble(model_list, data_loader, device, n_augmentations=10):
    """
    Optimized TTA that processes entire BATCHES instead of single sequences
    """
    all_predictions = []

    for batch_items in data_loader:
        X_batch = batch_items[0].to(device)  # (batch_size, 44, 35)
        static_batch = batch_items[1].to(device)  # (batch_size, 3)

        batch_size = X_batch.shape[0]

        # List to accumulate probabilities of all augmentations
        all_batch_probs = []

        # --- Iteration over models ---
        for model in model_list:
            model.eval()

            with torch.no_grad():
                # Original prediction (entire batch)
                logits = model(X_batch, static_batch)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                all_batch_probs.append(probs)

                # Temporal shifts (entire batch)
                for shift in [-3, -2, -1, 1, 2, 3]:
                    shifted_batch = X_batch.clone()

                    if shift > 0:
                        # Shift forward: moves data to the left
                        shifted_batch[:, :-shift] = X_batch[:, shift:]
                        shifted_batch[:, -shift:] = X_batch[:, -1:].repeat(1, shift, 1)
                    else:
                        # Shift backward: moves data to the right
                        shifted_batch[:, -shift:] = X_batch[:, :shift]
                        shifted_batch[:, :-shift] = X_batch[:, :1].repeat(1, -shift, 1)

                    logits = model(shifted_batch, static_batch)
                    probs = F.softmax(logits, dim=1).cpu().numpy()
                    all_batch_probs.append(probs)

                # Gaussian noise (entire batch)
                for _ in range(5):
                    noise = torch.randn_like(X_batch) * 0.006
                    noisy_batch = torch.clamp(X_batch + noise, 0, 1)

                    logits = model(noisy_batch, static_batch)
                    probs = F.softmax(logits, dim=1).cpu().numpy()
                    all_batch_probs.append(probs)

        # Average of all augmentations (for each sequence in the batch)
        # all_batch_probs has shape: (50, batch_size, 3)
        # where 50 = 5 models × 10 augmentations
        all_batch_probs = np.array(all_batch_probs)  # (50, batch_size, 3)
        avg_probs = np.mean(all_batch_probs, axis=0)  # (batch_size, 3)

        # Final predictions
        batch_preds = np.argmax(avg_probs, axis=1)  # (batch_size,)
        all_predictions.extend(batch_preds)

    return np.array(all_predictions)

## **Prepare the submission**

In [32]:
# Prepare the actual test set for submission
df_submission_test = pd.read_csv('pirate_pain_test.csv')

# Apply the same preprocessing steps as used for training and validation dataframes
# Map categorical columns
for col, mapping in zip(["n_legs", "n_hands", "n_eyes"], [legs_map, hands_map, eyes_map]):
    df_submission_test[col] = df_submission_test[col].str.strip().map(mapping)

# Scale numerical columns using min/max from training set
for col in scale_columns:
    if col in df_submission_test.columns: # Ensure the column exists
        if maxs[col] == mins[col]:
            df_submission_test[col] = 0.0
        else:
            df_submission_test[col] = (df_submission_test[col] - mins[col]) / (maxs[col] - mins[col])

# Simplified sequence building for test set, assuming consistent sequence length
def build_submission_sequences(df_test_data, window, stride, feature_cols):
    sequences = []
    static_feats_list = [] # List to store static features for each sequence
    sample_indices = []

    for sample_id in df_test_data['sample_index'].unique():
        # Extract features for the current sample
        current_sample_data = df_test_data[df_test_data['sample_index'] == sample_id]
        time_series_data = current_sample_data[feature_cols].values
        # Extract static features once per sample_index. They are constant across time.
        static_feats = current_sample_data[['n_legs', 'n_hands', 'n_eyes']].iloc[0].values

        current_idx = 0
        while current_idx + window <= len(time_series_data):
            sequences.append(time_series_data[current_idx:current_idx + window])
            static_feats_list.append(static_feats) # Add the static features for this sequence
            sample_indices.append(sample_id)
            current_idx += stride
    return (
        np.array(sequences, dtype=np.float32),
        np.array(static_feats_list, dtype=np.int64), # Return static features as the second array
        np.array(sample_indices)
    )

# Generate sequences for the submission test set
X_submission_seq, static_submission_seq, submission_sample_indices_repeated = build_submission_sequences(
    df_submission_test, WINDOW_SIZE, STRIDE, feature_columns
)

# DataLoader for the submission test set
submission_test_ds = TensorDataset(
    torch.from_numpy(X_submission_seq),
    torch.from_numpy(static_submission_seq) # Include static features here
)
submission_test_loader = make_loader(submission_test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# Predictions
print("Generating predictions using Soft Voting Ensemble...")
print("Generating predictions with Test-Time Augmentation...")
print("Creating 10 echoes per window × 5 models = 50 predictions per sequence")

all_preds_numeric = predict_with_tta_ensemble(
    model_list=all_best_models,
    data_loader=submission_test_loader,
    device=device,
    n_augmentations=10  # 1 original + 4 shifts + 5 noise
)

print(f"✅ Generated {len(all_preds_numeric)} predictions with TTA")

# Aggregate predictions per original sample_index using majority vote
predictions_df_agg = pd.DataFrame({
    'sample_index_sequence': submission_sample_indices_repeated,
    'predicted_label_numeric': all_preds_numeric
})

final_submission_labels = predictions_df_agg.groupby('sample_index_sequence')['predicted_label_numeric'].apply(
    lambda x: x.mode()[0]
).reset_index()

final_submission_labels.rename(columns={'sample_index_sequence': 'sample_index'}, inplace=True)

# Map numerical labels to original string labels
inv_label_mapping = {0: 'no_pain', 1: 'low_pain', 2: 'high_pain'}
final_submission_labels['label'] = final_submission_labels['predicted_label_numeric'].map(inv_label_mapping)

# Create final submission DataFrame
submission_df = final_submission_labels[['sample_index', 'label']]

# Save CSV
submission_df.to_csv('submission_ensemble.csv', index=False)
print("Submission pronta! ✅")

Generating predictions using Soft Voting Ensemble...
Generating predictions with Test-Time Augmentation...
Creating 10 echoes per window × 5 models = 50 predictions per sequence
✅ Generated 7944 predictions with TTA
Submission pronta! ✅
